# Testing YourBench — custom benchmark generation from your own documents

This notebook walks through the full YourBench loop:

1. **Install** YourBench (requires Python **3.12+**)
2. **Create a source document** (a short domain text — swap in your own PDF/textbook chapter)
3. **Write a config** (models, pipeline stages, output schema)
4. **Run the pipeline** → get a QA dataset
5. **Inspect the generated questions**
6. **Mini evaluation loop** — score a candidate model on the generated benchmark and log results with a timestamp (the "trackability" part)

**Prerequisites:** an API key for at least one OpenAI-compatible provider (OpenAI, OpenRouter, a local vLLM/Ollama endpoint, etc.). YourBench talks to models through OpenAI-compatible APIs, so OpenRouter is convenient if you want to mix providers.


In [1]:
import sys
assert sys.version_info >= (3, 12), f"YourBench requires Python 3.12+, you have {sys.version}"
print(sys.version)

3.13.1 (main, Jan  5 2025, 05:33:47) [Clang 19.1.6 ]


## 1. API keys

Set your key(s) here or in a `.env` file. The config below references them as `$VAR_NAME`.


In [4]:
import os, getpass

# Option A: OpenAI directly
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

# Option B (alternative): OpenRouter — lets you mix Anthropic / OpenAI / Google models
# os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OPENROUTER_API_KEY: ")

# Optional: only needed if you want to push the dataset to the Hugging Face Hub
# os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")

OPENAI_API_KEY:  ········


## 2. Source document

For a self-contained test we generate a short educational text on colorectal cancer staging (one of your example domains). **In real use, drop your PDFs / DOCX / HTML files into `documents/` instead** — YourBench parses them into Markdown automatically.

A note on scale: with a document this small you'll get a handful of questions. Quality and coverage improve with 10+ pages of real source material.


In [5]:
from pathlib import Path

workdir = Path("yourbench_demo")
docs_dir = workdir / "documents"
docs_dir.mkdir(parents=True, exist_ok=True)

sample_text = """# Colorectal Cancer: Staging and Treatment Principles

## TNM Staging

Colorectal cancer is staged with the TNM system. T describes depth of invasion of
the primary tumor: T1 invades the submucosa, T2 the muscularis propria, T3 extends
through the muscularis propria into pericolorectal tissues, and T4 invades the
visceral peritoneum (T4a) or adjacent organs (T4b). N describes regional lymph node
involvement: N0 means no nodal metastases, N1 means 1-3 positive nodes, and N2 means
4 or more positive nodes. M describes distant metastasis: M0 absent, M1 present,
with M1a limited to one organ, M1b in more than one organ, and M1c indicating
peritoneal metastases.

## Stage Grouping

Stage I disease is T1-T2 N0 M0. Stage II is T3-T4 N0 M0, subdivided into IIA (T3),
IIB (T4a), and IIC (T4b). Stage III is any T with nodal involvement (N1-N2, M0).
Stage IV is any T, any N with distant metastases (M1).

## Treatment by Stage

Stage I colon cancer is treated with surgical resection alone; adjuvant chemotherapy
is not indicated. Stage II treatment is resection, with adjuvant chemotherapy
considered only for high-risk features such as T4 tumors, fewer than 12 lymph nodes
examined, poorly differentiated histology, or lymphovascular invasion. Stage III
disease is treated with resection followed by adjuvant chemotherapy, most commonly
FOLFOX (5-fluorouracil, leucovorin, oxaliplatin) or CAPOX (capecitabine,
oxaliplatin) for 3 to 6 months. Stage IV management depends on resectability of
metastases: oligometastatic liver or lung disease may be cured with resection plus
perioperative chemotherapy, while unresectable disease is managed with systemic
therapy guided by molecular profiling (RAS/BRAF mutation status, MSI status,
HER2 amplification).

## Rectal Cancer Specifics

Locally advanced rectal cancer (T3-T4 or node-positive) is managed differently from
colon cancer: neoadjuvant therapy precedes surgery. Options include long-course
chemoradiotherapy (45-50.4 Gy with concurrent capecitabine) or short-course
radiotherapy (25 Gy in 5 fractions), increasingly delivered within total neoadjuvant
therapy (TNT) protocols that move all chemotherapy before surgery. Patients with a
clinical complete response after neoadjuvant therapy may be candidates for
nonoperative "watch-and-wait" management. Tumors with deficient mismatch repair
(dMMR) show high response rates to neoadjuvant immunotherapy with PD-1 blockade.
"""

(docs_dir / "crc_staging.md").write_text(sample_text)
print("Wrote", docs_dir / "crc_staging.md")

Wrote yourbench_demo/documents/crc_staging.md


## 3. Custom output schema (optional but recommended)

This is where you shape the benchmark format. Below: **multiple-choice questions**, which makes downstream scoring trivial and objective (no LLM-as-judge needed for the eval loop later).


In [6]:
schema_code = """
from pydantic import BaseModel, Field

class DataFormat(BaseModel):
    question: str = Field(description="A self-contained exam-style question answerable strictly from the source text")
    choices: list[str] = Field(description="Exactly 4 answer options, plausible and mutually exclusive")
    answer_index: int = Field(description="0-based index of the correct option in choices")
    explanation: str = Field(description="Brief justification citing the source text")
    difficulty: str = Field(description="easy, medium, or hard")
"""

(workdir / "mcq_schema.py").write_text(schema_code)
print("Wrote", workdir / "mcq_schema.py")

Wrote yourbench_demo/mcq_schema.py


## 4. Config

Key choices:
- `model_list` — the model(s) used to **generate** the benchmark. Use a strong model here; benchmark quality is capped by the generator.
- Pipeline stages: `ingestion → summarization → chunking → single_hop_question_generation → prepare_lighteval`.
- We save **locally** (no Hub push needed for a test).

To use OpenRouter or a local vLLM/Ollama instead, set `base_url` + the matching key (see the commented block).


In [10]:
import yaml

config = {
    "hf_configuration": {
        "hf_dataset_name": "crc-staging-bench",
        "local_saving": True,
        "local_dataset_dir": str(workdir / "dataset"),
        "push_to_hub": False,                     # <- disable Hub for all stages
    },
    "model_list": [
        {
            "model_name": "gpt-4o-mini",
            "api_key": "$OPENAI_API_KEY",
            "base_url": "https://api.openai.com/v1",
        }
    ],
    "pipeline": {
        "ingestion": {
            "source_documents_dir": str(docs_dir),
            "output_dir": str(workdir / "ingested"),
            "upload_to_hub": False,               # <- disable ingestion's own upload
        },
        "summarization": {},
        "chunking": {},
        "single_hop_question_generation": {
            "question_schema": str(workdir / "mcq_schema.py"),
        },
        "prepare_lighteval": {},
    },
}
config_path = workdir / "config.yaml"
config_path.write_text(yaml.dump(config, sort_keys=False))
print(config_path.read_text())

hf_configuration:
  hf_dataset_name: crc-staging-bench
  local_saving: true
  local_dataset_dir: yourbench_demo/dataset
  push_to_hub: false
model_list:
- model_name: gpt-4o-mini
  api_key: $OPENAI_API_KEY
  base_url: https://api.openai.com/v1
pipeline:
  ingestion:
    source_documents_dir: yourbench_demo/documents
    output_dir: yourbench_demo/ingested
    upload_to_hub: false
  summarization: {}
  chunking: {}
  single_hop_question_generation:
    question_schema: yourbench_demo/mcq_schema.py
  prepare_lighteval: {}



## 5. Validate and run

`yourbench validate` catches config errors cheaply; `yourbench estimate` previews token usage before you spend money. Note: config keys occasionally change between releases — if `validate` complains about a key (e.g. the local-saving options), run `yourbench init` to generate a known-good starter config for your installed version and diff against it.


In [11]:
!yourbench validate {config_path}


╭─────────────────────────────────────────────────────────────╮
│  YourBench - Dynamic Evaluation Set Generation           │
│  Build domain-specific benchmarks from your documents     │
╰─────────────────────────────────────────────────────────────╯

⠋ Validating configuration...
✓ Configuration is valid!

                             Configuration Summary                              
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Setting     ┃ Value                                                          ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Dataset     │ crc-staging-bench                                              │
│ Push to Hub │ ✗                                                              │
│ Private     │ ✗                                                              │
│ Models      │ gpt-4o-mini                                                    │
│ Stages      │ ingestion, summarization, 

In [12]:
!yourbench estimate {config_path}


╭─────────────────────────────────────────────────────────────╮
│  YourBench - Dynamic Evaluation Set Generation           │
│  Build domain-specific benchmarks from your documents     │
╰─────────────────────────────────────────────────────────────╯

⠋ Analyzing configuration...0m
Source Documents:
  Files: 2
  Estimated tokens: 1.3K

                           Token Estimation by Stage                            
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Stage           ┃ Input Tokens ┃ Output Tokens ┃ API Calls ┃ Notes           ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ Ingestion       │            - │             - │         - │ No LLM calls    │
│                 │              │               │           │ (text           │
│                 │              │               │           │ extraction      │
│                 │              │               │           │ only)           │
│ Summarizati

In [13]:
# The actual run — a few API calls, typically well under $1 for this tiny document
!yourbench run {config_path} --debug

12:17:57 | INFO     | Logging configured. JSON logs: logs/yourbench_20260820_121757.jsonl, Summary: logs/yourbench_20260820_121757_summary.log

╭─────────────────────────────────────────────────────────────╮
│  YourBench - Dynamic Evaluation Set Generation           │
│  Build domain-specific benchmarks from your documents     │
╰─────────────────────────────────────────────────────────────╯

⠋ Loading configuration...
                             Configuration Summary                              
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Setting     ┃ Value                                                          ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Dataset     │ crc-staging-bench                                              │
│ Push to Hub │ ✗                                                              │
│ Private     │ ✗                                                              │
│ Models  

## 6. Inspect the generated benchmark


In [17]:
from datasets import load_from_disk
import glob, json

# The pipeline writes one subset per stage; the questions live in the
# single-hop subset (and a lighteval-ready one if prepare_lighteval ran).
found = sorted(glob.glob(str(workdir / "dataset" / "**" / "dataset_info.json"), recursive=True))
print("Datasets written:")
for f in found:
    print(" -", f.rsplit("/", 2)[-2])

Datasets written:
 - chunked
 - ingested
 - prepared_lighteval
 - single_hop_questions
 - summarized


In [18]:
# Load the single-hop questions subset (adjust the path to what the cell above printed)
import os
candidates = [p for p in glob.glob(str(workdir / "dataset" / "*")) if os.path.isdir(p)]
qa_dir = next(p for p in candidates if "single" in p.lower() or "lighteval" in p.lower())
ds = load_from_disk(qa_dir)
print(ds)
ds = ds["train"] if hasattr(ds, "keys") and "train" in ds else ds

Dataset({
    features: ['question', 'additional_instructions', 'ground_truth_answer', 'gold', 'choices', 'question_category', 'kind', 'estimated_difficulty', 'citations', 'document_id', 'chunk_ids', 'question_generating_model', 'chunks', 'document', 'document_summary'],
    num_rows: 14
})


In [19]:
import json
for i, row in enumerate(ds.select(range(min(5, len(ds))))):
    print(f"--- Q{i+1} ---")
    print(json.dumps({k: row[k] for k in row.keys() if k in
        ("question", "choices", "answer_index", "explanation", "difficulty",
         "self_answer", "ground_truth_answer", "citations")}, indent=2, default=str)[:1500])
    print()

--- Q1 ---
{
  "question": "What does the 'T' in the TNM staging system for colorectal cancer represent?",
  "ground_truth_answer": "",
  "choices": [],
  "citations": []
}

--- Q2 ---
{
  "question": "Which stage of colorectal cancer requires surgery as the only treatment option?",
  "ground_truth_answer": "",
  "choices": [],
  "citations": []
}

--- Q3 ---
{
  "question": "What is the significant difference in the treatment of locally advanced rectal cancer compared to colon cancer?",
  "ground_truth_answer": "",
  "choices": [],
  "citations": []
}

--- Q4 ---
{
  "question": "How is Stage III colorectal cancer treated following surgery?",
  "ground_truth_answer": "",
  "choices": [],
  "citations": []
}

--- Q5 ---
{
  "question": "What does the 'M' in the TNM classification indicate?",
  "ground_truth_answer": "",
  "choices": [],
  "citations": []
}



## 7. Mini evaluation loop + results ledger

This is a deliberately simple stand-in for lighteval / lm-eval-harness / Inspect: it scores any OpenAI-compatible model on the generated MCQs by exact answer-letter match, and appends a row to a local **results ledger** — your `(model, benchmark, score, timestamp)` trackability requirement in ~40 lines.

Once you're past the toy stage, hand the `prepare_lighteval` output to lighteval instead — but this loop is enough to compare, say, `gpt-4o-mini` vs `gpt-4o`, or two quantizations of the same model served from a local endpoint.


In [25]:
import re
from openai import OpenAI

LETTERS = "ABCDEFGH"

def _gold_index(gold, choices):
    """gold can be [int], int, a letter, or the answer text."""
    g = gold[0] if isinstance(gold, list) and gold else gold
    if isinstance(g, int):
        return g
    if isinstance(g, str):
        s = g.strip()
        if len(s) == 1 and s.upper() in LETTERS:
            return LETTERS.index(s.upper())
        if s in choices:
            return choices.index(s)
    return None

def evaluate(model, dataset, base_url=None, api_key_env="OPENAI_API_KEY",
             judge_model="gpt-4o-mini"):
    client = OpenAI(base_url=base_url, api_key=os.environ[api_key_env])
    judge = OpenAI(api_key=os.environ["OPENAI_API_KEY"])  # judge stays on a fixed provider
    correct = total = 0

    for row in dataset:
        choices = row.get("choices") or []
        if choices:  # ---- MCQ path ----
            gold_idx = _gold_index(row.get("gold"), choices)
            if gold_idx is None:
                continue
            options = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(choices))
            prompt = (f"{row['question']}\n\n{options}\n\n"
                      "Answer with the single letter of the correct option and nothing else.")
            resp = client.chat.completions.create(model=model, max_tokens=5, temperature=0,
                messages=[{"role": "user", "content": prompt}])
            m = re.search(rf"[{LETTERS[:len(choices)]}]",
                          resp.choices[0].message.content.strip().upper())
            ok = m is not None and LETTERS.index(m.group()) == gold_idx
        else:        # ---- open-ended path: LLM-as-judge ----
            resp = client.chat.completions.create(model=model, max_tokens=300, temperature=0,
                messages=[{"role": "user", "content":
                    f"{row['question']}\n\nAnswer concisely."}])
            answer = resp.choices[0].message.content.strip()
            verdict = judge.chat.completions.create(model=judge_model, max_tokens=5, temperature=0,
                messages=[{"role": "user", "content":
                    "You are grading an exam answer.\n\n"
                    f"Question: {row['question']}\n\n"
                    f"Reference answer: {row['ground_truth_answer']}\n\n"
                    f"Candidate answer: {answer}\n\n"
                    "Does the candidate answer convey the same essential content as the "
                    "reference? Reply with exactly CORRECT or INCORRECT."}])
            ok = "CORRECT" in verdict.choices[0].message.content.strip().upper() \
                 and "INCORRECT" not in verdict.choices[0].message.content.strip().upper()
        correct += int(ok)
        total += 1
    return correct / total if total else float("nan"), total

In [26]:
print(qa_dir)
print(ds.column_names)

yourbench_demo/dataset/prepared_lighteval
['question', 'additional_instructions', 'ground_truth_answer', 'gold', 'choices', 'question_category', 'kind', 'estimated_difficulty', 'citations', 'document_id', 'chunk_ids', 'question_generating_model', 'chunks', 'document', 'document_summary']


In [27]:
DB = str(workdir / "results.sqlite")
BENCH = "crc-staging-bench-v1"

for model in ["gpt-4o-mini", "gpt-4o"]:          # <- swap in whatever you want to compare
    score, n = evaluate(model, ds)
    log_result(DB, model, BENCH, score, n)
    print(f"{model}: {score:.1%} on {n} questions")

gpt-4o-mini: 100.0% on 14 questions
gpt-4o: 92.9% on 14 questions


In [29]:
import pandas as pd, sqlite3
con = sqlite3.connect(DB)
pd.read_sql("SELECT * FROM results ORDER BY ts DESC", con)

,ts,model,benchmark,score,n_questions
0,2026-08-20T10:28:05.505864+00:00,gpt-4o,crc-staging-bench-v1,0.928571,14
1,2026-08-20T10:27:36.884572+00:00,gpt-4o-mini,crc-staging-bench-v1,1.000000,14
2,2026-08-20T10:23:09.767983+00:00,gpt-4o,crc-staging-bench-v1,NaN,0
3,2026-08-20T10:23:09.754477+00:00,gpt-4o-mini,crc-staging-bench-v1,NaN,0
4,2026-08-20T10:22:52.318731+00:00,gpt-4o,crc-staging-bench-v1,NaN,0
5,2026-08-20T10:22:52.311313+00:00,gpt-4o-mini,crc-staging-bench-v1,NaN,0


## Where to go next

- **Real sources**: replace the sample doc with a textbook chapter PDF. If the PDF has charts/figures, look at the `rich_pdf_extraction_with_gemini` example config in the repo.
- **Multi-hop questions**: add `multi_hop_question_generation:` to the pipeline for questions spanning multiple chunks — harder to benchmax against.
- **Quality gate**: add `citation_score_filtering:` to drop questions whose answers aren't grounded in the source, and always human-review a sample before trusting rankings (especially for healthcare).
- **Serious eval runner**: point `lighteval` at the `prepare_lighteval` output, or convert the dataset to an lm-eval-harness YAML task to test quantized local models (GGUF Q4 vs FP16 via llama.cpp/vLLM).
- **Ledger**: the SQLite table above is the seed of your trackability layer — add columns for provider, quantization, benchmark version (hash of the source docs + config), and generator model.
